# System Prompts — Practice

Companion to [../03_system_prompts.md](../03_system_prompts.md).

In [ ]:
#import dependencies
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

In [ ]:
#Define params
model = "claude-sonnet-5"

#define client
client = Anthropic()

In [ ]:
#helper functions
def add_user_message(messages, text):
    message = { "role": "user", "content": text}
    messages.append(message)

def add_assistant_message(messages, text):
    message = { "role": "assistant", "content": text}
    messages.append(message)

def chat(messages, system_prompt=None):
    params = {
        "model": model,
        "max_tokens": 1024,
        "messages": messages,
    }

    if system_prompt:
        params["system"] = system_prompt

    response = client.messages.create(**params)
    text_blocks = [block.text for block in response.content if block.type == "text"]
    return "\n".join(text_blocks)
    

In [ ]:
#pretty-print a turn as rendered Markdown -- wraps to the cell
#width and syntax-highlights code, instead of one long unwrapped line
from IPython.display import display, Markdown

def display_turn(role, text):
    display(Markdown(f"**{role}:**\n\n{text}"))

In [ ]:
messages = []
system_propmt = """
You are an expert in hindi and english.
Read the request from the user? and revert with approriate response.
Just tell what would the message be in hindi, no long definitions and extra information
Do not include the examples in response
Example:
What is hello in hind? Here user is expecting a response like below
Hello - नमस्ते/नमस्कार
"""

while True:
    user_input = input()
    display_turn("User", user_input)

    add_user_message(messages, user_input)

    response_text = chat(messages, system_propmt)

    add_assistant_message(messages, response_text)

    display_turn("Assistant", response_text)

## Example run

```
> User: hello
> Assistant: Hello - नमस्ते/नमस्कार
> User: how would I say water?
> Assistant: Water - पानी
> User: wanna see - I want to go to airport
> Assistant: I want to go to airport - मुझे एयरपोर्ट जाना है
```

Note the third turn: the system prompt shaped the output format
consistently across all three turns, and the model correctly
carried multi-turn context (translating a full sentence, not just
single words like the first two turns) -- both the system prompt
and the message history are doing their jobs together here.

## Next exercise: a system prompt for writing Python code

Write a system prompt that gets the model to consistently act as a
Python code generator -- think about what belongs in the prompt to
shape *how* it writes code, not just that it writes code:

- Output format -- code only, or code + a short explanation?
- Style conventions -- docstrings, type hints, comment density?
- Constraints -- Python version, standard library only vs. allowed
  to suggest packages, max function length, error handling
  expectations?
- What it should refuse or push back on, if anything?

Implementation is yours -- reuse the same chat() / message-history
pattern from the exercise above, just swap in a new system prompt.

In [ ]:
system_propmt = """
You are a python code expert. 
Who write simple and concise python code which compiles and executes successfully.
Code is intented to run on a python3 environment.
Do not fill the code with a sea of comments, use where a comment should be added for more clarity
Keep method names self explanatory
Use a modular approach to structure the code into methods
"""
messages = []
user_input = "Write a python program which checks a string for duplicate characters"
add_user_message(messages, user_input)

response_text = chat(messages, system_propmt)
display_turn("Assistant", response_text)

**Assistant:**

```python
def has_duplicate_characters(input_string):
    seen_characters = set()
    for char in input_string:
        if char in seen_characters:
            return True
        seen_characters.add(char)
    return False


def find_duplicate_characters(input_string):
    seen_characters = set()
    duplicates = set()
    for char in input_string:
        if char in seen_characters:
            duplicates.add(char)
        seen_characters.add(char)
    return duplicates


def main():
    test_string = input("Enter a string: ")

    if has_duplicate_characters(test_string):
        duplicates = find_duplicate_characters(test_string)
        print(f"Duplicate characters found: {duplicates}")
    else:
        print("No duplicate characters found.")


if __name__ == "__main__":
    main()
```

**Example run:**
```
Enter a string: programming
Duplicate characters found: {'r', 'g', 'm'}
```

The program uses two helper functions:
- `has_duplicate_characters`: quickly checks if any duplicates exist (early exit for efficiency).
- `find_duplicate_characters`: collects all characters that appear more than once.